# Lab 6: Governance Role Tests

This notebook demonstrates RLS and CLS using three roles:

- **admin:** all rows and full production values.
- **analyst:** all rows, but production values are masked.
- **regional_viewer:** only the assigned subregion, with masked production values.

The test role is selected using Databricks widgets.

**Note:** Role switching is performed for demonstration purposes only. In a production system users must not be allowed to change their own roles.

In [0]:
dbutils.widgets.text("catalog_name","dbr_dev_ua_5816_trail_2")
dbutils.widgets.text("gold_schema","viktoriia_kalenichenko_gold")
dbutils.widgets.dropdown("test_role","analyst",["admin", "analyst", "regional_viewer"])
dbutils.widgets.dropdown("test_subregion", "Eastern Europe",["Eastern Europe", "Northern Europe", "Southern Europe", "Western Europe"])

In [0]:
CATALOG = dbutils.widgets.get("catalog_name")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
DEMO = f"{GOLD}.governance_demo"
ROLES = f"{GOLD}.access_roles"

In [0]:

# setup widgets before update
test_role = dbutils.widgets.get("test_role")
test_subregion = dbutils.widgets.get("test_subregion")

# update the role for the current user (demo only)
spark.sql(f"""
    UPDATE {ROLES}
    SET
        role = '{test_role}',
        allowed_subregion = CASE
            WHEN '{test_role}' = 'regional_viewer'
            THEN '{test_subregion}'
            ELSE NULL
        END
    WHERE user_email = current_user()
""")

print(f"Demo role selected: {test_role}")
print(f"Allowed subregion: {test_subregion if test_role == 'regional_viewer' else 'All'}")

In [0]:
display(spark.sql(f"""SELECT country_name, subregion, year, total_production FROM {DEMO} ORDER BY country_name, year LIMIT 20"""))

display(spark.sql(f"""SELECT COUNT(*) AS visible_rows, COUNT(DISTINCT country_name) AS visible_countries, COUNT(DISTINCT subregion) AS visible_subregions
FROM {DEMO} """))